# Data Integrity Case Workbook

Read case document and complete all tasks in this workbook.


## Group Information

Write group name and all members' name in this cell.

- Group Name:
- Members:


## Import

In [ ]:
import sqlite3
import pandas as pd

pd.set_option('display.max_columns', None)


## Task 1: Load Transaction Data (5 points)
- Load 2020 and 2021 transaction data.
- Combine two years data to one DataFrame `df`.
- Explore `df` a bit. Display a few rows and check basic dataframe information.


In [ ]:
df_2020 = pd.read_excel('jinchange_2020.xlsx')
df_2021 = pd.read_excel('jinchange_2021.xlsx')

print('2020 shape:', df_2020.shape)
print('2021 shape:', df_2021.shape)

df = pd.concat([df_2020, df_2021], ignore_index=True)
df['Invoice_Date'] = pd.to_datetime(df['Invoice_Date'], errors='coerce')
df['Payment_Date'] = pd.to_datetime(df['Payment_Date'], errors='coerce')

df.head()


In [ ]:
df.info()
df.describe(include='number')


## Task 2: Create DB Tables (20 points)
- Create a sqlite database `jinchang.db`.
- Enable foreign key enforcement.
- Create tables with autoincrement primary keys and proper constraints.
- Drop the tables if they already exist.


In [ ]:
# Connect to (or create) a SQLite database file
conn = sqlite3.connect('jinchang.db')

# Enable foreign key constraints
conn.execute('PRAGMA foreign_keys = ON;')

print('Database ready')


## Create SQL Tables

In [ ]:
sql_script = """
DROP TABLE IF EXISTS payments;
DROP TABLE IF EXISTS shipping_handling;
DROP TABLE IF EXISTS purchase_invoices;
DROP TABLE IF EXISTS invoice_items;
DROP TABLE IF EXISTS sales_invoices;
DROP TABLE IF EXISTS suppliers;
DROP TABLE IF EXISTS products;
DROP TABLE IF EXISTS customers;

CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_name TEXT UNIQUE NOT NULL,
    customer_email TEXT,
    customer_address TEXT
);

CREATE TABLE products (
    product_id INTEGER PRIMARY KEY AUTOINCREMENT,
    model TEXT UNIQUE NOT NULL,
    model_description TEXT
);

CREATE TABLE suppliers (
    supplier_id INTEGER PRIMARY KEY AUTOINCREMENT,
    supplier_name TEXT UNIQUE NOT NULL,
    supplier_email TEXT,
    supplier_address TEXT
);

CREATE TABLE sales_invoices (
    invoice_id INTEGER PRIMARY KEY AUTOINCREMENT,
    invoice_number TEXT UNIQUE NOT NULL,
    trade_type TEXT NOT NULL CHECK (trade_type IN ('Export', 'Domestic')),
    invoice_date TEXT,
    customer_id INTEGER NOT NULL,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
);

CREATE TABLE invoice_items (
    item_id INTEGER PRIMARY KEY AUTOINCREMENT,
    invoice_id INTEGER NOT NULL,
    product_id INTEGER NOT NULL,
    quantity_ton REAL NOT NULL,
    unit_price_usd REAL DEFAULT 0,
    unit_price_rmb REAL DEFAULT 0,
    total_receivable_usd REAL DEFAULT 0,
    total_receivable_rmb REAL DEFAULT 0,
    FOREIGN KEY (invoice_id) REFERENCES sales_invoices(invoice_id),
    FOREIGN KEY (product_id) REFERENCES products(product_id)
);

CREATE TABLE shipping_handling (
    shipping_id INTEGER PRIMARY KEY AUTOINCREMENT,
    item_id INTEGER UNIQUE NOT NULL,
    ocean_freight_fee REAL DEFAULT 0,
    port_charge REAL DEFAULT 0,
    insurance REAL DEFAULT 0,
    land_freight REAL DEFAULT 0,
    other_charge REAL DEFAULT 0,
    courier_fee REAL DEFAULT 0,
    handling_fee REAL DEFAULT 0,
    FOREIGN KEY (item_id) REFERENCES invoice_items(item_id)
);

CREATE TABLE purchase_invoices (
    purchase_id INTEGER PRIMARY KEY AUTOINCREMENT,
    item_id INTEGER UNIQUE NOT NULL,
    supplier_id INTEGER NOT NULL,
    purchase_quantity_ton REAL NOT NULL,
    purchase_unit_price_rmb REAL DEFAULT 0,
    purchase_total_rmb REAL DEFAULT 0,
    sales_profit REAL DEFAULT 0,
    FOREIGN KEY (item_id) REFERENCES invoice_items(item_id),
    FOREIGN KEY (supplier_id) REFERENCES suppliers(supplier_id)
);

CREATE TABLE payments (
    payment_id INTEGER PRIMARY KEY AUTOINCREMENT,
    invoice_id INTEGER NOT NULL,
    payment_date TEXT,
    total_received_usd REAL DEFAULT 0,
    foreign_bank_deduction_usd REAL DEFAULT 0,
    settlement_rmb REAL DEFAULT 0,
    total_received_rmb REAL DEFAULT 0,
    FOREIGN KEY (invoice_id) REFERENCES sales_invoices(invoice_id)
);
"""

conn.executescript(sql_script)
print('Tables created')


## Task 3: Populate Data (20 points)
- Populate data from the spreadsheets to the DB tables.
- Commit changes to the DB.


In [ ]:
df.columns

df_customer = df[['Customer_Name', 'Customer_Email', 'Customer_Address']].drop_duplicates()
df_customer_to_db = df_customer.rename(columns={
    'Customer_Name': 'customer_name',
    'Customer_Email': 'customer_email',
    'Customer_Address': 'customer_address'
})
df_customer_to_db.to_sql('customers', conn, if_exists='append', index=False)

# Verify result
df_customer = pd.read_sql_query('select * from customers', conn)
df_customer.head()


In [ ]:
df_product = df[['Model', 'Model_Description']].drop_duplicates()
df_product_to_db = df_product.rename(columns={
    'Model': 'model',
    'Model_Description': 'model_description'
})
df_product_to_db.to_sql('products', conn, if_exists='append', index=False)

# Verify result
df_product = pd.read_sql_query('select * from products', conn)
df_product.head()


In [ ]:
df_supplier = df[['Supplier_Name', 'Supplier_Email', 'Supplier_Address']].drop_duplicates()
df_supplier_to_db = df_supplier.rename(columns={
    'Supplier_Name': 'supplier_name',
    'Supplier_Email': 'supplier_email',
    'Supplier_Address': 'supplier_address'
})
df_supplier_to_db.to_sql('suppliers', conn, if_exists='append', index=False)

# Verify result
df_supplier = pd.read_sql_query('select * from suppliers', conn)
df_supplier.head()


In [ ]:
df_invoice = df[['Invoice_Number', 'Trade_Type', 'Invoice_Date', 'Customer_Name']].copy()
df_invoice = df_invoice.sort_values(['Invoice_Number', 'Invoice_Date']).groupby('Invoice_Number', as_index=False).first()
df_invoice['Invoice_Date'] = df_invoice['Invoice_Date'].dt.strftime('%Y-%m-%d')
df_invoice = pd.merge(df_invoice, df_customer[['customer_id', 'customer_name']], left_on='Customer_Name', right_on='customer_name', how='left')

df_invoice_to_db = df_invoice[['Invoice_Number', 'Trade_Type', 'Invoice_Date', 'customer_id']].rename(columns={
    'Invoice_Number': 'invoice_number',
    'Trade_Type': 'trade_type',
    'Invoice_Date': 'invoice_date'
})
df_invoice_to_db.to_sql('sales_invoices', conn, if_exists='append', index=False)

payment_columns = ['Invoice_Number', 'Payment_Date', 'Total_Received_USD', 'Foreign_Bank_Deduction_USD', 'Settlement_RMB', 'Total_Received_RMB']
df_payment = df[payment_columns].copy()
payment_mask = df_payment['Payment_Date'].notna() | df_payment[['Total_Received_USD', 'Foreign_Bank_Deduction_USD', 'Settlement_RMB', 'Total_Received_RMB']].fillna(0).ne(0).any(axis=1)
df_payment = df_payment[payment_mask].drop_duplicates()
df_payment['Payment_Date'] = df_payment['Payment_Date'].dt.strftime('%Y-%m-%d')
df_payment = pd.merge(df_payment, pd.read_sql_query('select invoice_id, invoice_number from sales_invoices', conn), left_on='Invoice_Number', right_on='invoice_number', how='left')

df_payment_to_db = df_payment[['invoice_id', 'Payment_Date', 'Total_Received_USD', 'Foreign_Bank_Deduction_USD', 'Settlement_RMB', 'Total_Received_RMB']].rename(columns={
    'Payment_Date': 'payment_date',
    'Total_Received_USD': 'total_received_usd',
    'Foreign_Bank_Deduction_USD': 'foreign_bank_deduction_usd',
    'Settlement_RMB': 'settlement_rmb',
    'Total_Received_RMB': 'total_received_rmb'
})
df_payment_to_db.to_sql('payments', conn, if_exists='append', index=False)

pd.read_sql_query('select * from sales_invoices limit 5', conn)


In [ ]:
def insert_line_level_data(df_line_input, conn):
    df_invoice_id = pd.read_sql_query('select invoice_id, invoice_number from sales_invoices', conn)
    df_product_id = pd.read_sql_query('select product_id, model from products', conn)
    df_supplier_id = pd.read_sql_query('select supplier_id, supplier_name from suppliers', conn)

    df_line_ready = pd.merge(df_line_input, df_invoice_id, left_on='Invoice_Number', right_on='invoice_number', how='left')
    df_line_ready = pd.merge(df_line_ready, df_product_id, left_on='Model', right_on='model', how='left')
    df_line_ready = pd.merge(df_line_ready, df_supplier_id, left_on='Supplier_Name', right_on='supplier_name', how='left')

    for row in df_line_ready.itertuples(index=False):
        cursor = conn.execute(
            '''
            INSERT INTO invoice_items (
                invoice_id, product_id, quantity_ton, unit_price_usd,
                unit_price_rmb, total_receivable_usd, total_receivable_rmb
            ) VALUES (?, ?, ?, ?, ?, ?, ?)
            ''',
            (
                int(row.invoice_id), int(row.product_id), float(row.Quantity_Ton),
                float(row.Unit_Price_USD), float(row.Unit_Price_RMB),
                float(row.Total_Receivable_USD), float(row.Total_Receivable_RMB)
            )
        )
        item_id = cursor.lastrowid

        conn.execute(
            '''
            INSERT INTO shipping_handling (
                item_id, ocean_freight_fee, port_charge, insurance,
                land_freight, other_charge, courier_fee, handling_fee
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?)
            ''',
            (
                item_id, float(row.Ocean_Freight_Fee), float(row.Port_Charge),
                float(row.Insurance), float(row.Land_Freight), float(row.Other_Charge),
                float(row.Courier_Fee), float(row.Handling_Fee)
            )
        )

        conn.execute(
            '''
            INSERT INTO purchase_invoices (
                item_id, supplier_id, purchase_quantity_ton,
                purchase_unit_price_rmb, purchase_total_rmb, sales_profit
            ) VALUES (?, ?, ?, ?, ?, ?)
            ''',
            (
                item_id, int(row.supplier_id), float(row.Purchase_Invoice_Quantity_Ton),
                float(row.Purchase_Invoice_Unit_Price_RMB),
                float(row.Purchase_Invoice_Total_RMB), float(row.Sales_Profit)
            )
        )

line_columns = [
    'Invoice_Number', 'Model', 'Quantity_Ton', 'Unit_Price_USD', 'Unit_Price_RMB',
    'Total_Receivable_USD', 'Total_Receivable_RMB', 'Ocean_Freight_Fee', 'Port_Charge',
    'Insurance', 'Land_Freight', 'Other_Charge', 'Courier_Fee', 'Handling_Fee',
    'Supplier_Name', 'Purchase_Invoice_Quantity_Ton', 'Purchase_Invoice_Unit_Price_RMB',
    'Purchase_Invoice_Total_RMB', 'Sales_Profit'
]

df_line = df[line_columns].copy()
insert_line_level_data(df_line, conn)

conn.commit()
pd.read_sql_query('select * from invoice_items limit 5', conn)


In [ ]:
conn.close()


## Task 4: Add New Transaction (20 points)

- Describe the process to add a new sale.
- Record two new sales into the database. The new sales information is in `new_sales.csv`.
- Commit changes and close DB connection.


### Task 4-1: Describe the Process to Add a New Sale

- Step 1: Check whether the customer, product, and supplier already exist in the master tables.
- Step 2: Insert any missing customer, product, and supplier records.
- Step 3: Insert the invoice header once for each new invoice number.
- Step 4: Insert payment records for rows that contain payment information.
- Step 5: Insert one invoice item for each product line.
- Step 6: Insert the related shipping/handling and purchase invoice records.
- Step 7: Commit changes and verify the new invoices.


In [ ]:
conn = sqlite3.connect('jinchang.db')
conn.execute('PRAGMA foreign_keys = ON;')

df_new_sales = pd.read_csv('new_sales.csv')
df_new_sales['Invoice_Date'] = pd.to_datetime(df_new_sales['Invoice_Date'], errors='coerce')
df_new_sales['Payment_Date'] = pd.to_datetime(df_new_sales['Payment_Date'], errors='coerce')
df_new_sales

# Add missing customers
df_new_customer = df_new_sales[['Customer_Name', 'Customer_Email', 'Customer_Address']].drop_duplicates()
existing_customer = pd.read_sql_query('select customer_name from customers', conn)
df_new_customer = df_new_customer[~df_new_customer['Customer_Name'].isin(existing_customer['customer_name'])]
if not df_new_customer.empty:
    df_new_customer.rename(columns={
        'Customer_Name': 'customer_name',
        'Customer_Email': 'customer_email',
        'Customer_Address': 'customer_address'
    }).to_sql('customers', conn, if_exists='append', index=False)

# Add missing products
df_new_product = df_new_sales[['Model', 'Model_Description']].drop_duplicates()
existing_product = pd.read_sql_query('select model from products', conn)
df_new_product = df_new_product[~df_new_product['Model'].isin(existing_product['model'])]
if not df_new_product.empty:
    df_new_product.rename(columns={
        'Model': 'model',
        'Model_Description': 'model_description'
    }).to_sql('products', conn, if_exists='append', index=False)

# Add missing suppliers
df_new_supplier = df_new_sales[['Supplier_Name', 'Supplier_Email', 'Supplier_Address']].drop_duplicates()
existing_supplier = pd.read_sql_query('select supplier_name from suppliers', conn)
df_new_supplier = df_new_supplier[~df_new_supplier['Supplier_Name'].isin(existing_supplier['supplier_name'])]
if not df_new_supplier.empty:
    df_new_supplier.rename(columns={
        'Supplier_Name': 'supplier_name',
        'Supplier_Email': 'supplier_email',
        'Supplier_Address': 'supplier_address'
    }).to_sql('suppliers', conn, if_exists='append', index=False)

# Add new invoice headers
df_new_invoice = df_new_sales[['Invoice_Number', 'Trade_Type', 'Invoice_Date', 'Customer_Name']].copy()
df_new_invoice = df_new_invoice.sort_values(['Invoice_Number', 'Invoice_Date']).groupby('Invoice_Number', as_index=False).first()
df_new_invoice = pd.merge(df_new_invoice, pd.read_sql_query('select customer_id, customer_name from customers', conn), left_on='Customer_Name', right_on='customer_name', how='left')
df_new_invoice['Invoice_Date'] = df_new_invoice['Invoice_Date'].dt.strftime('%Y-%m-%d')
df_new_invoice_to_db = df_new_invoice[['Invoice_Number', 'Trade_Type', 'Invoice_Date', 'customer_id']].rename(columns={
    'Invoice_Number': 'invoice_number',
    'Trade_Type': 'trade_type',
    'Invoice_Date': 'invoice_date'
})
df_new_invoice_to_db.to_sql('sales_invoices', conn, if_exists='append', index=False)

# Add payment records
df_new_payment = df_new_sales[payment_columns].copy()
payment_mask = df_new_payment['Payment_Date'].notna() | df_new_payment[['Total_Received_USD', 'Foreign_Bank_Deduction_USD', 'Settlement_RMB', 'Total_Received_RMB']].fillna(0).ne(0).any(axis=1)
df_new_payment = df_new_payment[payment_mask].drop_duplicates()
df_new_payment['Payment_Date'] = df_new_payment['Payment_Date'].dt.strftime('%Y-%m-%d')
df_new_payment = pd.merge(df_new_payment, pd.read_sql_query('select invoice_id, invoice_number from sales_invoices', conn), left_on='Invoice_Number', right_on='invoice_number', how='left')
df_new_payment_to_db = df_new_payment[['invoice_id', 'Payment_Date', 'Total_Received_USD', 'Foreign_Bank_Deduction_USD', 'Settlement_RMB', 'Total_Received_RMB']].rename(columns={
    'Payment_Date': 'payment_date',
    'Total_Received_USD': 'total_received_usd',
    'Foreign_Bank_Deduction_USD': 'foreign_bank_deduction_usd',
    'Settlement_RMB': 'settlement_rmb',
    'Total_Received_RMB': 'total_received_rmb'
})
df_new_payment_to_db.to_sql('payments', conn, if_exists='append', index=False)

# Add line-level records
df_new_line = df_new_sales[line_columns].copy()
insert_line_level_data(df_new_line, conn)

conn.commit()
pd.read_sql_query("select * from sales_invoices where invoice_number in ('25-HCCH001', '25-HCCH002')", conn)


## Task 5: Data Analysis (15 points)

- Connect to Jinchang Database.
- Load data to DataFrames.
- Complete the required analyses.


In [ ]:
conn.close()
conn = sqlite3.connect('jinchang.db')

df_customer = pd.read_sql_query('select * from customers', conn)
df_product = pd.read_sql_query('select * from products', conn)
df_supplier = pd.read_sql_query('select * from suppliers', conn)
df_invoice = pd.read_sql_query('select * from sales_invoices', conn)
df_item = pd.read_sql_query('select * from invoice_items', conn)
df_purchase = pd.read_sql_query('select * from purchase_invoices', conn)
df_payment = pd.read_sql_query('select * from payments', conn)

df_invoice_all = pd.merge(df_invoice, df_customer, on='customer_id', how='left')
df_payment_all = pd.merge(df_payment, df_invoice_all, on='invoice_id', how='left')

df_item_all = pd.merge(df_item, df_product, on='product_id', how='left')
df_item_all = pd.merge(df_item_all, df_purchase, on='item_id', how='left')
df_item_all = pd.merge(df_item_all, df_supplier, on='supplier_id', how='left')


### Analysis #1: Top 5 International Customers by Sales
- Rank international customers by total sales (`settlement_rmb`).
- List top 5 customer names and total settlement in RMB.


In [ ]:
df_payment_all[df_payment_all['trade_type'] == 'Export']         .groupby('customer_name', as_index=False)['settlement_rmb']         .sum()         .sort_values('settlement_rmb', ascending=False)         .head(5)         .rename(columns={'settlement_rmb': 'total_settlement_rmb'})


### Analysis #2: Top 5 Products by Quantity Sold
- Rank products by quantity sold.
- List top 5 product models and total quantity sold.


In [ ]:
df_item_all.groupby('model', as_index=False)['quantity_ton']         .sum()         .sort_values('quantity_ton', ascending=False)         .head(5)         .rename(columns={'quantity_ton': 'total_quantity_ton'})


### Analysis #3: Top 5 Suppliers by Total Purchasing Amount
- Rank suppliers by purchasing amount.
- List top 5 supplier names and total purchasing amount.


In [ ]:
df_item_all.groupby('supplier_name', as_index=False)['purchase_total_rmb']         .sum()         .sort_values('purchase_total_rmb', ascending=False)         .head(5)         .rename(columns={'purchase_total_rmb': 'total_purchase_rmb'})


### Close DB Connection

In [ ]:
conn.close()